# Download read pairs

The k-mer counts are calculated from the same number of read pairs for every sequencing run. For data available through ENA, `download_read_pairs` takes this fixed number from the two FASTQ files listed in an ENA run report. The report contains `run_accession`, which identifies the run, and `fastq_ftp`, which gives the URLs for read 1 and read 2 separated by a semicolon.

`download_read_pairs` checks the structure of each FASTQ record while it downloads the two files and, after both downloads finish, compares the read identifiers between read 1 and read 2. Once every run has been downloaded, `download_read_pairs` writes `fastq_manifest.tsv` with the run accession, the absolute paths of the two FASTQ files, and the number of read pairs. `count_kmers` reads this FASTQ manifest to count the canonical k-mers for each run.

## Set the paths and options

Before running the notebook, install `fastq_classifier` from the repository root with `python -m pip install .` in the Python environment used by the Jupyter kernel. `DATA_DIR` must point to the downloaded reproduction data, which contains `ena_runs.tsv`. The notebook writes `ena_runs_example.tsv` and the downloaded FASTQ files to `WORK_DIR`; `WORK_DIR` is set to `work` in this example.

In [ ]:
import csv
from itertools import islice
from pathlib import Path

from fastq_classifier import download_read_pairs

DATA_DIR = Path("path/to/reproduction_data_v1")
ENA_REPORT = DATA_DIR / "ena_runs.tsv"

WORK_DIR = Path("work")
EXAMPLE_REPORT = WORK_DIR / "ena_runs_example.tsv"
DOWNLOAD_DIR = WORK_DIR / "fastq"

RUNS_TO_DOWNLOAD = 2
READ_PAIRS = 25_000
JOBS = 4
RUN_DOWNLOAD = False

## Choose a small set of runs

`ena_runs.tsv` contains one row for each of the 4,401 sequencing runs used to build the development count matrix. Calling `download_read_pairs` with this complete ENA run report would download `READ_PAIRS` read pairs for all 4,401 runs. To keep the example small, the code below copies the first `RUNS_TO_DOWNLOAD` rows to `ena_runs_example.tsv`; `RUNS_TO_DOWNLOAD` is set to `2` here. Set `RUNS_TO_DOWNLOAD` to the number of sequencing runs that you want to download.

In [ ]:
with ENA_REPORT.open(encoding="utf-8-sig", newline="") as source:
    example_run_rows = list(
        islice(csv.DictReader(source, delimiter="\t"), RUNS_TO_DOWNLOAD)
    )

WORK_DIR.mkdir(parents=True, exist_ok=True)
with EXAMPLE_REPORT.open("w", encoding="utf-8", newline="") as destination:
    writer = csv.DictWriter(
        destination,
        fieldnames=["run_accession", "fastq_ftp"],
        delimiter="\t",
    )
    writer.writeheader()
    writer.writerows(example_run_rows)

print("Runs selected:", [row["run_accession"] for row in example_run_rows])
print("Example report:", EXAMPLE_REPORT)

## Download the reads

`RUN_DOWNLOAD` is set to `False` so that running the notebook does not immediately start a download. First check the ENA run accessions written to `ena_runs_example.tsv` and the download directory given by `DOWNLOAD_DIR`. Then set `RUN_DOWNLOAD` to `True` and run the cell below. `READ_PAIRS` sets the number of read pairs downloaded from each sequencing run, while `JOBS` sets the number of sequencing runs downloaded at the same time. An internet connection is required to download the FASTQ files.

In [ ]:
if RUN_DOWNLOAD:
    manifest_path = download_read_pairs(
        EXAMPLE_REPORT,
        DOWNLOAD_DIR,
        read_pairs=READ_PAIRS,
        jobs=JOBS,
    )
    print("FASTQ manifest:", manifest_path)
else:
    print("Download skipped. Set RUN_DOWNLOAD = True when you want to fetch the reads.")

## Inspect the FASTQ manifest

After `download_read_pairs` finishes, `fastq_manifest.tsv` contains one row for each sequencing run in `ena_runs_example.tsv`:

| Column | Contents |
|---|---|
| `run_accession` | ENA run accession |
| `read1_path` | Absolute path to the read 1 FASTQ file |
| `read2_path` | Absolute path to the read 2 FASTQ file |
| `read_pairs` | Number of records kept from each mate |

`read_pairs` has the same value in every row because `count_kmers` requires every sequencing run in a FASTQ manifest to contain the same number of read pairs. When `fastq_manifest.tsv` is passed to `count_kmers`, `count_kmers` records `read_pairs` in each `run.json` file and in `kmc_manifest.tsv`. Notebook 02 shows how to call `count_kmers` with `fastq_manifest.tsv` and read the resulting `kmc_manifest.tsv`.

In [ ]:
manifest_path = DOWNLOAD_DIR / "fastq_manifest.tsv"
if manifest_path.exists():
    with manifest_path.open(encoding="utf-8", newline="") as stream:
        for manifest_row in csv.DictReader(stream, delimiter="\t"):
            print(manifest_row)
else:
    print("No manifest found. Run the download cell first.")